# Transform Races Data

1. Read bronze `races` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`raceName` → `race_name`, `circuitId` → `circuit_id`)
1. Rename columns to make them more meaningful (`date` → `race_date`)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `races` table

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
races_drop_col_df = races_df.drop("url")

In [0]:
races_renamed_col_df = races_drop_col_df.withColumnsRenamed(
    {"raceName": "race_name", "circuitId": "circuit_id", "date": "race_date"}
)

In [0]:
races_drop_duplicates_df = races_renamed_col_df.dropDuplicates(["round", "season"])

In [0]:
races_final_df = races_drop_duplicates_df.withColumn(
    "race_name", F.initcap(F.col("race_name"))
).withColumn("circuit_id", F.initcap(F.col("circuit_id")))

In [0]:
(races_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table))

In [0]:
%sql
select
  *
from
  formula1.silver.races;